In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

df     = pd.read_csv("../data/processed/master_df.csv", index_col=0, parse_dates=True)
events = pd.read_csv("../data/raw/geopolitical_events.csv", parse_dates=["date"])

WINDOW = 30  # days before and after event

In [3]:
# ── CALCULATE CUMULATIVE RETURNS AROUND EACH EVENT ─────
def event_window(df, event_date, col, window=WINDOW):
    """Returns cumulative % change from event date baseline"""
    start = event_date - pd.Timedelta(days=window)
    end   = event_date + pd.Timedelta(days=window)
    sub   = df[col].loc[start:end].dropna()
    if len(sub) < 5:
        return None
    baseline = sub.loc[sub.index <= event_date].iloc[-1]  # value at event
    cum_ret  = ((sub - baseline) / baseline) * 100         # % change from event
    # Normalize index to days from event
    days = [(d - event_date).days for d in sub.index]
    return pd.Series(cum_ret.values, index=days)

# Assets to study
assets = {
    "USDINR": ("USD/INR (↑=INR weakens)", "#D62828"),
    "GOLD":   ("Gold Price",               "#F4A261"),
    "CRUDE":  ("Crude Oil",                "#2A9D8F"),
    "NIFTY":  ("Nifty 50",                 "#1A3A5C"),
}

In [4]:
# ── SUMMARY TABLE ───────────────────────────────────────
print("=" * 70)
print(f"{'Event':<35} {'INR +5d':>8} {'Gold +5d':>8} {'Crude +5d':>9}")
print("=" * 70)
for _, ev in events.iterrows():
    row_vals = [ev["event"][:34]]
    for col in ["USDINR","GOLD","CRUDE"]:
        ew = event_window(df, ev["date"], col)
        if ew is not None and 5 in ew.index:
            row_vals.append(f"{ew[5]:+.2f}%")
        else:
            row_vals.append("  N/A")
    print(f"{row_vals[0]:<35} {row_vals[1]:>8} {row_vals[2]:>8} {row_vals[3]:>9}")
print("=" * 70)

Event                                INR +5d Gold +5d Crude +5d
Uri Surgical Strikes                  +0.07%   -4.19%    +1.80%
Pulwama Attack                        +1.18%   +2.31%    +3.09%
Balakot Airstrike                        N/A      N/A       N/A
Galwan Valley Clash                      N/A      N/A       N/A
Operation Sindoor                     +0.73%   -4.77%    +6.68%
Gulf of Oman Tanker Attacks           -0.25%   -0.87%    +1.78%
Saudi Aramco Drone Attack             +0.36%   +0.50%    +5.98%
Soleimani Killing                     +1.04%   +0.53%    -5.46%
Israel-Hamas War                      -0.01%   +2.14%    +0.14%
Houthi Red Sea Attacks                +0.27%   +1.04%    -0.46%
Russia-Ukraine War Begins             +0.87%   +0.90%   +11.42%
OPEC+ Production Cuts                 +1.68%   -2.58%    +3.84%
China Stock Market Crash                 N/A      N/A       N/A
COVID Global Lockdown                    N/A      N/A       N/A


In [5]:
# ── PLOTLY CHART: EVENT STUDY ───────────────────────────
fig = make_subplots(
    rows=len(events), cols=1,
    subplot_titles=[f"{r['event']} ({r['date'].strftime('%d %b %Y')})" 
                    for _, r in events.iterrows()],
    vertical_spacing=0.06,
    shared_xaxes=True
)

for i, (_, ev) in enumerate(events.iterrows()):
    row = i + 1
    for col, (label, color) in assets.items():
        ew = event_window(df, ev["date"], col)
        if ew is None:
            continue
        fig.add_trace(
            go.Scatter(
                x=list(ew.index), y=list(ew.values),
                mode="lines", name=label,
                line=dict(color=color, width=1.5),
                showlegend=(i == 0)
            ),
            row=row, col=1
        )
    # Zero line (event day)
    fig.add_hline(y=0, line_dash="dash", line_color="black", 
                  line_width=0.8, row=row, col=1)
    # Event day marker
    fig.add_vline(x=0, line_dash="dot", line_color="gray", 
                  line_width=1, row=row, col=1)

fig.update_layout(
    height=250 * len(events),
    title_text="<b>RupeeRisk — Geopolitical Event Study</b><br>"
               "<sub>Cumulative % change from event date | USD/INR, Gold, Crude, Nifty</sub>",
    template="plotly_white",
    font=dict(family="Arial", size=11),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0)
)
fig.update_xaxes(title_text="Days from Event", row=len(events), col=1)
fig.write_html("../data/processed/event_study.html")
fig.show()
print("Event study saved as interactive HTML.")

Event study saved as interactive HTML.


In [6]:
# ── CHANNEL ATTRIBUTION: Which channel did each event work through? ──

channel_assets = {
    "Direct FX":              ["USDINR", "INDIAVIX"],
    "Oil Supply":             ["CRUDE", "GOLD", "USDINR"],
    "Oil Supply + Risk-Off":  ["CRUDE", "USDINR", "NIFTY", "DXY"],
    "Risk-Off + DXY":         ["DXY", "NIFTY", "USDINR"],
    "Oil + Gas + Risk-Off + DXY": ["CRUDE", "DXY", "NIFTY", "USDINR"],
    "Trade Route + Oil":      ["CRUDE", "USDINR"],
    "Risk-Off + Oil Collapse":["CRUDE", "NIFTY", "USDINR"],
    "Direct FX + Trade":      ["USDINR", "NIFTY"],
}

print("\n" + "="*80)
print(f"{'Event':<32} {'Channel':<22} {'Crude+5d':>8} {'DXY+5d':>7} {'INR+5d':>7} {'Nifty+5d':>9}")
print("="*80)

attribution_rows = []
for _, ev in events.iterrows():
    row = {"Event": ev["event"], "Date": ev["date"].strftime("%d %b %Y"),
           "Channel": ev["channel"], "Category": ev["category"]}
    
    for col, label in [("CRUDE","Crude+5d"),("DXY","DXY+5d"),
                        ("USDINR","INR+5d"),("NIFTY","Nifty+5d")]:
        ew = event_window(df, ev["date"], col)   # reuse your existing function
        if ew is not None:
            val = ew.get(5, ew.get(4, ew.get(3, None)))
            row[label] = round(val, 2) if val is not None else None
        else:
            row[label] = None
    
    attribution_rows.append(row)
    print(f"{ev['event'][:31]:<32} {ev['channel'][:21]:<22} "
          f"{str(row.get('Crude+5d','N/A')):>8} {str(row.get('DXY+5d','N/A')):>7} "
          f"{str(row.get('INR+5d','N/A')):>7} {str(row.get('Nifty+5d','N/A')):>9}")

attribution_df = pd.DataFrame(attribution_rows)
attribution_df.to_csv("../data/processed/channel_attribution.csv", index=False)
print("="*80)

# ── CHANNEL COMPARISON: Average INR impact by channel type ──
print("\n=== AVERAGE INR DEPRECIATION BY CHANNEL ===")
channel_impact = attribution_df.groupby("Channel")["INR+5d"].mean().sort_values(ascending=False)
for ch, val in channel_impact.items():
    direction = "↑ INR weakens" if val > 0 else "↓ INR strengthens"
    print(f"  {ch:<35} {val:+.2f}%  {direction}")


Event                            Channel                Crude+5d  DXY+5d  INR+5d  Nifty+5d
Uri Surgical Strikes             Direct FX                   1.8    0.61    0.07      2.07
Pulwama Attack                   Direct FX                  3.09   -0.47    1.18     -1.32
Balakot Airstrike                Direct FX                  0.54    0.55   -0.39      0.26
Galwan Valley Clash              Direct FX + Trade          7.09    0.97    0.58      4.39
Operation Sindoor                Direct FX                  6.68    2.19    0.73      2.09
Gulf of Oman Tanker Attacks      Oil Supply                 1.78    0.69   -0.25      1.14
Saudi Aramco Drone Attack        Oil Supply                 5.98    0.01    0.36     -3.35
Soleimani Killing                Oil Supply + Risk-Off     -5.46    0.48    1.04     -1.65
Israel-Hamas War                 Oil Supply + Risk-Off      0.14     0.5   -0.01      0.71
Houthi Red Sea Attacks           Trade Route + Oil         -0.46    -0.4    0.27      0.3